# 02 — Tier 2 Live Pipeline VAD Benchmark (Jitsi Bridge)

This notebook provides the interactive test harness for **Tier 2 (Live Pipeline)** evaluation across a live Jitsi Meet / JVB room.

### Public Ingress Topology
**Implemented Ingress Architecture**: **Cloudflare Tunnel / Cloudflare Worker WebSocket Gateway** (with `cloudflared` / `ngrok` fallback for local development).
- `bots/listener_bot.py` acts as the WebSocket Ingest Server accepting media forwarded from the Jitsi bridge.
- The tunnel or Cloudflare Worker endpoint provides public reachability into ephemeral notebook runtimes (Kaggle / Google Colab).

### Tier 2 Workflow
1. **SpeakerBot** (`bots/speaker_bot.py`) joins the Jitsi room and injects manifest audio clips with embedded calibration chirps.
2. **ListenerBot** (`bots/listener_bot.py`) receives Opus packets from the bridge feed, timestamps arrival, decodes to PCM, resamples to 16kHz mono, and tracks packet loss.
3. **Marker Alignment** (`latency/marker_alignment.py`) establishes physical "Time Zero" ($T_0$) on the Listener Bot's clock via normalized cross-correlation.
4. **Latency Calculator** (`latency/latency_calculator.py`) calculates separate Onset Latency and Offset / Hangover Latency.
5. **Scorer & Storage** (`scoring/vad_scorer.py`, `storage/results_store.py`) execute identical scoring and persistent incremental writes as Tier 1.

> **Boundary Rule**: This notebook contains zero scoring, latency math, or Jitsi protocol logic. It delegates execution entirely to `orchestrator.py`.

In [ ]:
# Cell 1: Environment Setup & Project Path Configuration
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root configured at: {project_root}")

In [ ]:
# Cell 2: Public Ingress Gateway & Tunnel Configuration
import os

# Public WebSocket Ingress URL (Cloudflare Worker endpoint or Tunnel URL)
# Configured per the existing Jitsi bridge connection blueprint
CLOUDFLARE_WORKER_GATEWAY_URL = os.environ.get(
    "JITSI_BRIDGE_URL",
    "wss://jitsi-bridge-gateway.benchmark-infra.workers.dev/audio-feed"
)
JITSI_ROOM_NAME = os.environ.get(
    "JITSI_ROOM_NAME",
    "vad-benchmark-live-eval-01"
)

print(f"Ingress Gateway URL: {CLOUDFLARE_WORKER_GATEWAY_URL}")
print(f"Target Jitsi Room:   {JITSI_ROOM_NAME}")

In [ ]:
# Cell 3: Kaggle/Colab Session Time-Boxing & Tier 2 Batch Configuration
import json
import yaml
from corpus.manifest import load_manifest

# Session Time-Boxing Parameters (defending against Kaggle kernel preemption)
MAX_CLIPS_PER_BATCH = 15          # Bounded clip batch size to survive short session windows
WARMUP_CLIPS_TIER2 = 2           # Discard initial clips to absorb Jitsi WebRTC connection handshake
CONCURRENCY_MODE = "sequential"  # 'sequential' for strict latency timing; 'parallel' for accuracy-only
CATEGORY_FILTER = []             # e.g., ['clean speech', 'packet-loss/jitter'] or [] for all

tier2_work_dir = project_root / "results" / "tier2_session"
tier2_work_dir.mkdir(parents=True, exist_ok=True)

# Prepare declarative Tier 2 run configuration
tier2_config = {
    "run_id": "tier2_live_eval_20260913_001",
    "tiers_to_run": ["tier2"],
    "active_adapters": ["ten_vad", "firered_vad", "webrtc_vad"],
    "webrtc_aggressiveness_modes": [0, 1, 2, 3],
    "corpus_manifest_path": "corpus/manifest.json",
    "corpus_category_filter": CATEGORY_FILTER,
    "hardware_policy": "cpu_only",
    "warmup_clip_count": WARMUP_CLIPS_TIER2,
    "scoring_grid_ms": 10,
    "concurrency_mode": CONCURRENCY_MODE,
    "include_held_out_slice": False,
    "results_destination": str(tier2_work_dir),
    "jitsi_room": JITSI_ROOM_NAME,
    "bridge_url": CLOUDFLARE_WORKER_GATEWAY_URL
}

tier2_config_path = tier2_work_dir / "run_config_tier2.yaml"
with open(tier2_config_path, "w", encoding="utf-8") as f:
    yaml.dump(tier2_config, f, indent=2)

print(f"Tier 2 time-boxed run configuration saved to: {tier2_config_path}")

In [ ]:
# Cell 4: The Single Substantive Call into orchestrator.py for Tier 2 Execution
from orchestrator import run_benchmark

run_summary = run_benchmark(config_path=tier2_config_path)
print("\n--- Tier 2 Execution Summary ---")
print(json.dumps(run_summary, indent=2))

In [ ]:
# Cell 5: Display Live Pipeline Latency & VAD Accuracy Results from ResultsStore
from storage.results_store import ResultsStore

store = ResultsStore(tier2_work_dir)
records = list(store.read_raw_records(run_id=tier2_config["run_id"]))

print(f"Total Tier 2 persistent evaluation records loaded: {len(records)}\n")

# 1. Display Tier 2 Latency Records (Onset Latency & Offset/Hangover Latency)
print("=== TIER 2 PIPELINE LATENCY MEASUREMENTS ===")
latency_rows = [r for r in records if r.get("record_type") == "tier2_pipeline_latency"]
for r in latency_rows[:8]:
    p = r["payload"]
    mean_onset = f"{p['mean_onset_latency_ms']:.1f} ms" if p.get('mean_onset_latency_ms') is not None else "Undefined (FN)"
    mean_offset = f"{p['mean_offset_hangover_latency_ms']:.1f} ms" if p.get('mean_offset_hangover_latency_ms') is not None else "Undefined (Hangover)"
    print(f"[Adapter: {r['adapter_id']:<18}] [Clip: {r['clip_id']:<20}] "
          f"Onset Latency: {mean_onset:<16} | Offset Latency: {mean_offset:<16}")

# 2. Display Tier 2 Accuracy Records
print("\n=== TIER 2 VAD ACCURACY EVALUATION ===")
scoring_rows = [r for r in records if r.get("record_type") == "scoring_clip"]
for r in scoring_rows[:8]:
    m = r["payload"]["metrics"]
    print(f"[Adapter: {r['adapter_id']:<18}] [Category: {r['payload'].get('category', 'N/A'):<22}] "
          f"F1: {m['f1']:.3f} | Precision: {m['precision']:.3f} | Recall: {m['recall']:.3f} | FPR: {m['false_positive_rate']:.3f}")

store.close()